In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize, LinearSegmentedColormap
import matplotlib.cm as cm
import seaborn as sns
from datetime import datetime
import math
import glob

###########################################
# Configuration Parameters
###########################################

# Define runtime configuration
RUNTIME_CONFIG = {
    'DATA_FILES_PATTERN': 'JC2024*citibiketripdata.csv',  # Pattern to match all 2024 monthly files
    'DATA_DIR': 'data/citibike',  # Directory containing the Citibike trip data files
    'OUTPUT_DIR': 'results/citibike_od_annual',  # Output directory for results
    'MODE': 'visualize',  # Mode: 'train', 'predict', 'visualize', 'enhanced-viz', 'animate', 'all'
    'USE_GPU': True,  # Use GPU if available
    'MAX_STATIONS': 50,  # Maximum number of stations to include
    'TRAINING_EPOCHS': 100,  # Number of training epochs
    'SAMPLE_RATE': 1,  # Sampling rate for data loading - reduced to handle more data
    'DISTANCE_THRESHOLD': 2.0,  # Distance threshold for adjacency (km)
    # 'MAX_SAMPLES_PER_MONTH': 50000,  # Maximum samples to take from each month for memory efficiency
    'ANALYZE_SEASONAL_PATTERNS': True,  # Enable seasonal pattern analysis
}

# Citibike configuration parameters
CITIBIKE_CONFIG = {
    # Dataset characteristics
    'num_stations': RUNTIME_CONFIG['MAX_STATIONS'],
    'time_periods': 24,         # Hours of the day
    'day_types': 2,             # Weekday and weekend
    'user_types': 2,            # Member and casual riders
    
    # Architecture - Memory efficiency settings
    'gcn_hidden_dim': 128,      # Hidden dimensions for GCN
    'spatial_hidden_dim': 16,   # Hidden dimension for spatial features
    'temporal_hidden_dim': 24,  # Hidden dimension for temporal features
    'latent_dim': 64,           # Latent dimension for the model
    'dropout_rate': 0.3,        # Dropout rate to prevent overfitting
    'l2_reg': 1e-5,             # L2 regularization
    
    # Training parameters
    'batch_size': 64,           # Batch size
    'epochs': RUNTIME_CONFIG['TRAINING_EPOCHS'],
    'learning_rate': 1e-3,      # Initial learning rate
    'lr_decay_factor': 0.5,     # Learning rate decay factor
    'lr_patience': 5,           # Patience for learning rate scheduler
    'early_stopping_patience': 15, # Early stopping patience
    
    # Data paths
    'data_path': RUNTIME_CONFIG['DATA_DIR'],
    'output_path': RUNTIME_CONFIG['OUTPUT_DIR'],
    
    # Temporal features
    'peak_morning_start': 7,    # Morning peak hour start (7 AM)
    'peak_morning_end': 10,     # Morning peak hour end (10 AM)
    'peak_evening_start': 16,   # Evening peak hour start (4 PM)
    'peak_evening_end': 19,     # Evening peak hour end (7 PM)
    
    # Trip characteristics
    'avg_trip_duration': 7.39,  # Average trip duration in minutes
    'median_trip_duration': 5.37, # Median trip duration in minutes
    'avg_station_distance': 3.77, # Average distance between stations in km
    
    # Feature selection
    'use_distance_features': True,  # Use station distances
    'use_temporal_features': True,  # Use time of day, day of week
    'enable_attention': False,      # Disable attention to save memory
    
    # Visualization settings
    'max_stations_viz': 50,     # Max stations for visualizations
    'sample_rate_viz': RUNTIME_CONFIG['SAMPLE_RATE'],  # Sampling rate for visualizations
    'animation_max_stations': 12,  # Max stations for animations
    'animation_sample_rate': 0.1,  # Sampling rate for animations
    
    # Seasonal analysis
    'analyze_seasons': RUNTIME_CONFIG['ANALYZE_SEASONAL_PATTERNS'],    # Enable seasonal trend analysis
    'seasons': {
        'winter': [1, 2, 12],    # Jan, Feb, Dec
        'spring': [3, 4, 5],     # Mar, Apr, May
        'summer': [6, 7, 8],     # Jun, Jul, Aug
        'fall': [9, 10, 11]      # Sep, Oct, Nov
    },
    
    # Weekend vs Weekday analysis
    'analyze_day_types': True,   # Enable weekday/weekend comparison
}

###########################################
# Helper Functions
###########################################

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate haversine distance between two points in kilometers"""
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371  # Radius of earth in kilometers
    
    return c * r

def create_temporal_features(hour, day_of_week, month, config):
    """Create temporal features for the model"""
    # Cyclical encoding for hour of day (sine and cosine)
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    
    # Cyclical encoding for day of week
    day_sin = np.sin(2 * np.pi * day_of_week / 7)
    day_cos = np.cos(2 * np.pi * day_of_week / 7)
    
    # Cyclical encoding for month of year
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)
    
    # Peak hour indicators
    is_morning_peak = 1 if config['peak_morning_start'] <= hour < config['peak_morning_end'] else 0
    is_evening_peak = 1 if config['peak_evening_start'] <= hour < config['peak_evening_end'] else 0
    
    # Weekend indicator
    is_weekend = 1 if day_of_week >= 5 else 0  # 5=Saturday, 6=Sunday
    
    # Seasonal indicators (using month)
    seasons = config.get('seasons', {})
    is_winter = 1 if month in seasons.get('winter', [12, 1, 2]) else 0
    is_spring = 1 if month in seasons.get('spring', [3, 4, 5]) else 0
    is_summer = 1 if month in seasons.get('summer', [6, 7, 8]) else 0
    is_fall = 1 if month in seasons.get('fall', [9, 10, 11]) else 0
    
    return [hour_sin, hour_cos, day_sin, day_cos, month_sin, month_cos,
            is_morning_peak, is_evening_peak, is_weekend,
            is_winter, is_spring, is_summer, is_fall]

def prepare_adjacency_matrix(distance_matrix, threshold=2.0):
    """Prepare adjacency matrix for GCN based on station distances"""
    num_stations = distance_matrix.shape[0]
    adj_matrix = np.zeros((num_stations, num_stations))
    
    # Create adjacency based on distance threshold
    for i in range(num_stations):
        for j in range(num_stations):
            if i != j and distance_matrix[i, j] <= threshold:
                adj_matrix[i, j] = 1
    
    # Add self-loops
    adj_matrix = adj_matrix + np.eye(num_stations)
    
    # Normalize adjacency matrix
    rowsum = np.array(adj_matrix.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    
    normalized_adj = adj_matrix.dot(d_mat_inv_sqrt).transpose().dot(d_mat_inv_sqrt)
    
    return normalized_adj

# def load_citibike_data(data_files_pattern, sample_rate=0.2, max_samples_per_month=50000):
#     """Load and preprocess Citibike data from multiple files"""
#     print(f"Loading Citibike data with {sample_rate*100:.1f}% sampling rate")
    
#     # Find all matching files
#     data_dir = os.path.dirname(data_files_pattern)
#     if not data_dir:
#         data_dir = '.'
#     file_list = glob.glob(data_files_pattern)
    
#     if not file_list:
#         raise FileNotFoundError(f"No data files found matching pattern: {data_files_pattern}")
    
#     file_list.sort()  # Ensure files are processed in order
#     print(f"Found {len(file_list)} files to process")
    
#     # Load data from each file
#     all_trips = []
    
#     for file_path in file_list:
#         file_name = os.path.basename(file_path)
#         print(f"Processing file: {file_name}")
        
#         # Extract month from filename (assuming format JC2024MMcitibiketripdata.csv)
#         try:
#             month = int(file_name[6:8])
#             print(f"Extracted month: {month}")
#         except (ValueError, IndexError):
#             month = 0
#             print(f"Could not extract month from filename: {file_name}, using placeholder")
        
#         # Load data in chunks to save memory
#         chunks = []
        
#         for chunk in pd.read_csv(file_path, chunksize=10000):
#             # Sample the chunk to reduce memory usage
#             sampled_chunk = chunk.sample(frac=sample_rate, random_state=42)
#             chunks.append(sampled_chunk)
            
#             # Check if we've reached the maximum samples for this month
#             if sum(len(c) for c in chunks) >= max_samples_per_month:
#                 print(f"Reached maximum samples ({max_samples_per_month}) for {file_name}")
#                 break
        
#         # Combine chunks for this month
#         if chunks:
#             monthly_trips = pd.concat(chunks)
#             print(f"Loaded {len(monthly_trips)} sampled trips from {file_name}")
            
#             # Add month column if not already present
#             if 'month' not in monthly_trips.columns:
#                 monthly_trips['month'] = month
            
#             all_trips.append(monthly_trips)
#         else:
#             print(f"No data loaded from {file_name}")

def load_citibike_data(data_files_pattern, sample_rate=0.2):
    """Load and preprocess Citibike data from multiple files"""
    print(f"Loading Citibike data with {sample_rate*100:.1f}% sampling rate")
    
    # Find all matching files
    data_dir = os.path.dirname(data_files_pattern)
    if not data_dir:
        data_dir = '.'
    file_list = glob.glob(data_files_pattern)
    
    if not file_list:
        raise FileNotFoundError(f"No data files found matching pattern: {data_files_pattern}")
    
    file_list.sort() # Ensure files are processed in order
    print(f"Found {len(file_list)} files to process")
    
    # Load data from each file
    all_trips = []
    
    for file_path in file_list:
        file_name = os.path.basename(file_path)
        print(f"Processing file: {file_name}")
        
        # Extract month from filename (assuming format JC2024MMcitibiketripdata.csv)
        try:
            month = int(file_name[6:8])
            print(f"Extracted month: {month}")
        except (ValueError, IndexError):
            month = 0
            print(f"Could not extract month from filename: {file_name}, using placeholder")
        
        # Load data in chunks to save memory
        chunks = []
        
        for chunk in pd.read_csv(file_path, chunksize=10000):
            # Sample the chunk to reduce memory usage
            sampled_chunk = chunk.sample(frac=sample_rate, random_state=42)
            chunks.append(sampled_chunk)
            
            # Remove the check for maximum samples
        
        # Combine chunks for this month
        if chunks:
            monthly_trips = pd.concat(chunks)
            print(f"Loaded {len(monthly_trips)} sampled trips from {file_name}")
            
            # Add month column if not already present
            if 'month' not in monthly_trips.columns:
                monthly_trips['month'] = month
            
            all_trips.append(monthly_trips)
        else:
            print(f"No data loaded from {file_name}")
    # Combine all monthly data
    if not all_trips:
        raise ValueError("No trip data was loaded from any file")
    
    trips_df = pd.concat(all_trips)
    print(f"Combined dataset contains {len(trips_df)} trips")
    
    # Basic data cleaning
    # Remove trips with missing station information
    trips_df = trips_df.dropna(subset=['start_station_name', 'end_station_name', 
                                      'start_lat', 'start_lng', 
                                      'end_lat', 'end_lng'])
    
    # Convert timestamps to datetime - use format='mixed' to handle various timestamp formats
    print("Converting timestamps to datetime...")
    trips_df['started_at'] = pd.to_datetime(trips_df['started_at'], format='mixed')
    trips_df['ended_at'] = pd.to_datetime(trips_df['ended_at'], format='mixed')
    
    # Extract time features
    trips_df['start_hour'] = trips_df['started_at'].dt.hour
    trips_df['start_day'] = trips_df['started_at'].dt.dayofweek  # Monday=0, Sunday=6
    trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Make sure month is available
    if 'month' not in trips_df.columns:
        trips_df['month'] = trips_df['started_at'].dt.month
    
    # Calculate trip duration in minutes
    trips_df['duration_minutes'] = (trips_df['ended_at'] - trips_df['started_at']).dt.total_seconds() / 60
    
    # Filter out unrealistic trips (e.g., maintenance-related trips or errors)
    trips_df = trips_df[(trips_df['duration_minutes'] > 0) & 
                       (trips_df['duration_minutes'] < 24*60)]  # Less than 24 hours
    
    print(f"After cleaning: {len(trips_df)} trips")
    
    return trips_df

def get_top_stations(trips_df, max_stations):
    """Get the top stations by trip count and create station mapping"""
    # Get unique stations
    start_counts = trips_df['start_station_name'].value_counts()
    end_counts = trips_df['end_station_name'].value_counts()
    combined_counts = start_counts.add(end_counts, fill_value=0)
    
    # Take the top stations
    max_stations = min(max_stations, len(combined_counts))
    top_stations = combined_counts.sort_values(ascending=False).head(max_stations).index.tolist()
    
    print(f"Using top {len(top_stations)} stations")
    
    # Create station mapping
    station_mapping = {station: idx for idx, station in enumerate(top_stations)}
    
    # Create station data dictionary
    station_data = {}
    
    for station in top_stations:
        # Get station coordinates from the first occurrence
        start_rows = trips_df[trips_df['start_station_name'] == station]
        end_rows = trips_df[trips_df['end_station_name'] == station]
        
        if not start_rows.empty:
            lat = start_rows['start_lat'].iloc[0]
            lng = start_rows['start_lng'].iloc[0]
        elif not end_rows.empty:
            lat = end_rows['end_lat'].iloc[0]
            lng = end_rows['end_lng'].iloc[0]
        else:
            lat, lng = 0, 0
            print(f"No coordinate data found for station: {station}")
            
        # Calculate station popularity
        popularity = (
            trips_df['start_station_name'].value_counts().get(station, 0) +
            trips_df['end_station_name'].value_counts().get(station, 0)
        )
        
        station_data[station_mapping[station]] = {
            'name': station,
            'lat': lat,
            'lng': lng,
            'popularity': popularity
        }
    
    return station_mapping, station_data, top_stations

def calculate_distance_matrix(station_data):
    """Calculate distance matrix between stations"""
    num_stations = len(station_data)
    distance_matrix = np.zeros((num_stations, num_stations))
    
    for i in range(num_stations):
        for j in range(num_stations):
            if i != j:
                station_i = station_data[i]
                station_j = station_data[j]
                
                # Calculate haversine distance
                distance_matrix[i, j] = haversine_distance(
                    station_i['lat'], station_i['lng'],
                    station_j['lat'], station_j['lng']
                )
    
    return distance_matrix

def create_od_matrices(trips_df, station_mapping, num_stations):
    """Create origin-destination matrices for different time periods"""
    # Create 4D matrix: [month, day_type, hour, origin, destination]
    hours = 24
    day_types = 2  # Weekday and weekend
    months = 12  # All months of the year
    
    od_matrices = np.zeros((months, day_types, hours, num_stations, num_stations))
    
    # Filter trips to only include mapped stations
    mapped_trips = trips_df[
        trips_df['start_station_name'].isin(station_mapping.keys()) & 
        trips_df['end_station_name'].isin(station_mapping.keys())
    ]
    
    # Group trips by month, day type, hour, origin, and destination
    for month in range(1, months+1):
        month_trips = mapped_trips[mapped_trips['month'] == month]
        
        if month_trips.empty:
            print(f"No trips found for month {month}")
            continue
        
        print(f"Processing month {month} with {len(month_trips)} trips")
        
        for day_type in range(day_types):  # 0=weekday, 1=weekend
            is_weekend = day_type  # 0=False, 1=True
            
            day_trips = month_trips[month_trips['is_weekend'] == is_weekend]
            
            for hour in range(hours):
                hour_trips = day_trips[day_trips['start_hour'] == hour]
                
                for _, trip in hour_trips.iterrows():
                    origin_idx = station_mapping.get(trip['start_station_name'])
                    dest_idx = station_mapping.get(trip['end_station_name'])
                    
                    if origin_idx is not None and dest_idx is not None:
                        od_matrices[month-1, day_type, hour, origin_idx, dest_idx] += 1
    
    print(f"Created OD matrices of shape {od_matrices.shape}")
    print(f"Total trips in OD matrices: {np.sum(od_matrices)}")
    
    return od_matrices

def create_features_and_targets(od_matrices, station_data, distance_matrix, config):
    """Create features and targets for the model"""
    months, day_types, hours, num_stations, _ = od_matrices.shape
    
    features = []
    targets = []
    
    # Process data for each month
    for month in range(months):
        for day_type in range(day_types):
            for hour in range(hours):
                # Process every other hour for memory efficiency
                if hour % 2 == 0:
                    for origin in range(num_stations):
                        for dest in range(num_stations):
                            # Skip self-loops
                            if origin == dest:
                                continue
                            
                            # Get the flow value (target)
                            flow = od_matrices[month, day_type, hour, origin, dest]
                            
                            # Only include non-zero flows to reduce dataset size
                            if flow > 0:
                                # 1. Origin station attributes
                                origin_features = [
                                    station_data[origin]['lat'],
                                    station_data[origin]['lng'],
                                    station_data[origin]['popularity'] / 1000,
                                    1,  # Is origin indicator
                                    0   # Is destination indicator
                                ]
                                
                                # 2. Destination station attributes
                                dest_features = [
                                    station_data[dest]['lat'],
                                    station_data[dest]['lng'],
                                    station_data[dest]['popularity'] / 1000,
                                    0,  # Is origin indicator
                                    1   # Is destination indicator
                                ]
                                
                                # 3. All stations attributes (simplified to top 5 stations by distance)
                                station_features = []
                                
                                # Get distances from origin to all stations
                                distances = [(i, distance_matrix[origin, i]) 
                                           for i in range(num_stations) if i != origin and i != dest]
                                # Sort by distance and take top 5
                                top_stations = sorted(distances, key=lambda x: x[1])[:min(5, len(distances))]
                                
                                for station_idx, dist in top_stations:
                                    station_features.extend([
                                        station_data[station_idx]['lat'],
                                        station_data[station_idx]['lng'],
                                        station_data[station_idx]['popularity'] / 1000,
                                        dist
                                    ])
                                
                                # Pad if we have fewer than 5 stations
                                while len(top_stations) < 5:
                                    station_features.extend([0, 0, 0, 0])
                                    top_stations.append((None, None))
                                
                                # 4. Spatial features
                                spatial_features = [
                                    distance_matrix[origin, dest],  # Distance between origin and destination
                                    station_data[origin]['lat'] - station_data[dest]['lat'],  # Lat difference
                                    station_data[origin]['lng'] - station_data[dest]['lng'],  # Lng difference
                                ]
                                
                                # 5. Temporal features
                                temp_features = create_temporal_features(hour, day_type*5, month+1, config)
                                
                                # Combine all features
                                all_features = (
                                    origin_features + 
                                    dest_features + 
                                    station_features + 
                                    spatial_features + 
                                    temp_features
                                )
                                
                                features.append(all_features)
                                targets.append(flow)
    
    return np.array(features), np.array(targets)

###########################################
# Dataset and Model Classes
###########################################

class CitibikeODDataset(Dataset):
    """Custom Dataset for Citibike Origin-Destination Matrix data"""
    
    def __init__(self, features, targets):
        self.features = torch.FloatTensor(features)
        self.targets = torch.FloatTensor(targets)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

class GCNLayer(nn.Module):
    """Graph Convolutional Network Layer"""
    
    def __init__(self, in_features, out_features, activation=F.relu, dropout_rate=0.3):
        super(GCNLayer, self).__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.activation = activation
        self.dropout = nn.Dropout(dropout_rate) if dropout_rate > 0 else None
    
    def forward(self, inputs, adj_matrix):
        # Linear transformation
        hidden = self.linear(inputs)
        
        # Apply graph convolution: each node aggregates features from its neighbors
        batch_size, num_nodes, _ = hidden.shape
        
        for i in range(batch_size):
            hidden[i] = torch.matmul(adj_matrix, hidden[i])
        
        # Apply activation function
        if self.activation is not None:
            hidden = self.activation(hidden)
        
        # Apply dropout
        if self.dropout is not None:
            hidden = self.dropout(hidden)
        
        return hidden

class CitibikeODModel(nn.Module):
    """Model for Citibike Origin-Destination Matrix Estimation"""
    
    def __init__(self, config, feature_dim, normalized_adj):
        super(CitibikeODModel, self).__init__()
        
        self.config = config
        self.feature_dim = feature_dim
        
        # Register normalized adjacency matrix as buffer
        self.register_buffer('normalized_adj', torch.FloatTensor(normalized_adj))
        
        # Dimensions
        num_stations = config['num_stations']
        gcn_hidden_dim = config['gcn_hidden_dim']
        spatial_hidden_dim = config['spatial_hidden_dim']
        temporal_hidden_dim = config['temporal_hidden_dim']
        latent_dim = config['latent_dim']
        
        # Divide features into categories
        self.feature_categories = {
            'station_features': int(feature_dim * 0.5),  # 50% for station features
            'spatial_features': int(feature_dim * 0.2),  # 20% for spatial features
            'temporal_features': feature_dim - int(feature_dim * 0.5) - int(feature_dim * 0.2)  # 30% for temporal features
        }
        
        # Feature encoders
        self.station_encoder = nn.Sequential(
            nn.Linear(self.feature_categories['station_features'], gcn_hidden_dim),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate'])
        )
        
        self.spatial_encoder = nn.Sequential(
            nn.Linear(self.feature_categories['spatial_features'], spatial_hidden_dim),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate'])
        )
        
        self.temporal_encoder = nn.Sequential(
            nn.Linear(self.feature_categories['temporal_features'], temporal_hidden_dim),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate'])
        )
        
        # GCN for station relationships
        self.gcn = GCNLayer(
            in_features=gcn_hidden_dim,
            out_features=gcn_hidden_dim,
            dropout_rate=config['dropout_rate']
        )
        
        # Feature fusion - concatenate encoded features
        self.fusion_dim = gcn_hidden_dim + spatial_hidden_dim + temporal_hidden_dim
        
        # MLP for final prediction
        self.mlp = nn.Sequential(
            nn.Linear(self.fusion_dim, latent_dim),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate']),
            nn.Linear(latent_dim, latent_dim // 2),
            nn.ReLU(),
            nn.Dropout(config['dropout_rate']),
            nn.Linear(latent_dim // 2, 1)
        )
    
    def forward(self, x):
        batch_size = x.shape[0]
        
        # Split input by feature categories
        station_features = x[:, :self.feature_categories['station_features']]
        spatial_features = x[:, self.feature_categories['station_features']:
                            self.feature_categories['station_features'] + 
                            self.feature_categories['spatial_features']]
        temporal_features = x[:, self.feature_categories['station_features'] + 
                             self.feature_categories['spatial_features']:]
        
        # Encode features
        encoded_station = self.station_encoder(station_features)
        encoded_spatial = self.spatial_encoder(spatial_features)
        encoded_temporal = self.temporal_encoder(temporal_features)
        
        # Reshape station features for GCN (memory-efficient version)
        encoded_station_reshaped = encoded_station.unsqueeze(1)  # [batch_size, 1, gcn_hidden_dim]
        
        # Apply GCN - simplified to avoid memory issues
        gcn_out = self.gcn(encoded_station_reshaped, self.normalized_adj[:1, :1])
        gcn_out = gcn_out.squeeze(1)  # [batch_size, gcn_hidden_dim]
        
        # Concatenate all encoded features
        fusion = torch.cat([gcn_out, encoded_spatial, encoded_temporal], dim=1)
        
        # Final prediction
        output = self.mlp(fusion)
        
        return output.squeeze()

###########################################
# Training and Evaluation Functions
###########################################

def train_model(model, X_train, y_train, X_val, y_val, config, device='cuda'):
    """Train the Citibike OD model"""
    print("Training Citibike OD model...")
    
    # Create data loaders
    train_dataset = CitibikeODDataset(X_train, y_train)
    val_dataset = CitibikeODDataset(X_val, y_val)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config['batch_size'], 
        shuffle=True
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=config['batch_size'], 
        shuffle=False
    )
    
    # Define optimizer
    optimizer = Adam(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['l2_reg']
    )
    
    # Define learning rate scheduler (removed verbose=True parameter)
    scheduler = ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=config['lr_decay_factor'], 
        patience=config['lr_patience']
    )
    
    # Training history
    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}
    
    # Training loop
    best_val_loss = float('inf')
    early_stopping_counter = 0
    best_model_state = None
    
    for epoch in range(config['epochs']):
        # Training phase
        model.train()
        train_losses = []
        train_maes = []
        
        # Process each batch
        for features, targets in train_loader:
            # Move to device
            features = features.to(device)
            targets = targets.to(device)
            
            # Forward pass
            outputs = model(features)
            
            # Compute loss (MSE)
            loss = F.mse_loss(outputs, targets)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Track metrics
            train_losses.append(loss.item())
            train_mae = F.l1_loss(outputs, targets).item()
            train_maes.append(train_mae)
        
        # Calculate average training metrics
        avg_train_loss = sum(train_losses) / len(train_losses)
        avg_train_mae = sum(train_maes) / len(train_maes)
        
        # Validation phase
        model.eval()
        val_losses = []
        val_maes = []
        
        with torch.no_grad():
            for features, targets in val_loader:
                # Move to device
                features = features.to(device)
                targets = targets.to(device)
                
                # Forward pass
                outputs = model(features)
                
                # Compute loss
                loss = F.mse_loss(outputs, targets)
                
                # Track metrics
                val_losses.append(loss.item())
                val_mae = F.l1_loss(outputs, targets).item()
                val_maes.append(val_mae)
        
        # Calculate average validation metrics
        avg_val_loss = sum(val_losses) / len(val_losses)
        avg_val_mae = sum(val_maes) / len(val_maes)
        
        # Update learning rate scheduler
        scheduler.step(avg_val_loss)
        
        # Print current learning rate after scheduler update
        current_lr = optimizer.param_groups[0]['lr']
        
        # Save metrics
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_mae'].append(avg_train_mae)
        history['val_mae'].append(avg_val_mae)
        
        # Print epoch summary with learning rate
        print(f"Epoch {epoch+1}/{config['epochs']} - "
              f"Train Loss: {avg_train_loss:.4f}, Train MAE: {avg_train_mae:.4f}, "
              f"Val Loss: {avg_val_loss:.4f}, Val MAE: {avg_val_mae:.4f}, "
              f"LR: {current_lr:.6f}")
        
        # Check if current model is the best
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict().copy()
            early_stopping_counter = 0
            
            # Save best model
            model_path = os.path.join(config['output_path'], 'best_model.pth')
            print(f"Saving best model to {model_path}")
            torch.save({
                'epoch': epoch,
                'model_state_dict': best_model_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
                'config': config
            }, model_path)
        else:
            early_stopping_counter += 1
        
        # Early stopping
        if early_stopping_counter >= config['early_stopping_patience']:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, history

def predict(model, X, device='cuda'):
    """Make predictions using the trained model"""
    # Convert to tensor and move to device
    X_tensor = torch.FloatTensor(X).to(device)
    
    # Set model to evaluation mode
    model.eval()
    
    # Make predictions
    with torch.no_grad():
        predictions = model(X_tensor).cpu().numpy()
    
    return predictions

def evaluate_model(model, X, y, device='cuda'):
    """Evaluate the model on test data"""
    # Make predictions
    predictions = predict(model, X, device)
    
    # Calculate metrics
    mse = mean_squared_error(y, predictions)
    mae = mean_absolute_error(y, predictions)
    r2 = r2_score(y, predictions)
    
    # Print metrics
    print(f"Evaluation metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"RÂ²: {r2:.4f}")
    
    return {
        'mse': mse,
        'mae': mae,
        'r2': r2
    }

def calculate_additional_metrics(y_true, y_pred):
    """
    Calculate additional evaluation metrics for OD matrix prediction
    
    Parameters:
    -----------
    y_true : array-like
        Ground truth values
    y_pred : array-like
        Predicted values
        
    Returns:
    --------
    metrics : dict
        Dictionary of evaluation metrics
    """
    # Ensure numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Filter out zero values for MAPE calculation
    non_zero_mask = y_true != 0
    y_true_nz = y_true[non_zero_mask]
    y_pred_nz = y_pred[non_zero_mask]
    
    # Calculate existing metrics
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    # Calculate additional metrics
    # Normalized RMSE (NRMSE) - scale-independent measure
    nrmse = rmse / np.mean(y_true) if np.mean(y_true) > 0 else np.nan
    
    # Mean Absolute Percentage Error (MAPE)
    mape = np.mean(np.abs((y_true_nz - y_pred_nz) / y_true_nz)) * 100 if len(y_true_nz) > 0 else np.nan
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'NRMSE': nrmse,
        'MAPE': mape
    }

def create_metrics_table(metrics, output_dir, filename='metrics_table.txt'):
    """
    Create a formatted table of metrics
    
    Parameters:
    -----------
    metrics : dict
        Dictionary of evaluation metrics
    output_dir : str
        Directory to save the table
    filename : str
        Filename for the table
    """
    output_path = os.path.join(output_dir, filename)
    
    # Format metrics for table
    with open(output_path, 'w') as f:
        f.write("============================================================================================\n")
        f.write("                            Origin-Destination Model Evaluation Metrics                     \n")
        f.write("============================================================================================\n")
        f.write(f"{'Metric':<20} {'Value':<15} {'Description':<45}\n")
        f.write("--------------------------------------------------------------------------------------------\n")
        f.write(f"{'MAE':<20} {metrics['MAE']:<15.4f} {'Mean Absolute Error':<45}\n")
        f.write(f"{'MSE':<20} {metrics['MSE']:<15.4f} {'Mean Squared Error':<45}\n")
        f.write(f"{'RMSE':<20} {metrics['RMSE']:<15.4f} {'Root Mean Squared Error':<45}\n")
        f.write(f"{'RÂ²':<20} {metrics['R2']:<15.4f} {'Coefficient of Determination':<45}\n")
        f.write(f"{'NRMSE (%)':<20} {metrics['NRMSE']:<15.4f} {'Normalized RMSE (RMSE/mean)':<45}\n")
        f.write(f"{'MAPE (%)':<20} {metrics['MAPE']:<15.4f} {'Mean Absolute Percentage Error':<45}\n")
        f.write("============================================================================================\n")
    
    print(f"Metrics table saved to {output_path}")
    
    # Create LaTeX version for academic papers
    latex_path = os.path.join(output_dir, 'metrics_table.tex')
    with open(latex_path, 'w') as f:
        f.write("\\begin{table}[ht]\n")
        f.write("\\centering\n")
        f.write("\\caption{Origin-Destination Model Evaluation Metrics}\n")
        f.write("\\begin{tabular}{lll}\n")
        f.write("\\hline\n")
        f.write("Metric & Value & Description \\\\\n")
        f.write("\\hline\n")
        f.write(f"MAE & {metrics['MAE']:.4f} & Mean Absolute Error \\\\\n")
        f.write(f"MSE & {metrics['MSE']:.4f} & Mean Squared Error \\\\\n")
        f.write(f"RMSE & {metrics['RMSE']:.4f} & Root Mean Squared Error \\\\\n")
        f.write(f"R$^2$ & {metrics['R2']:.4f} & Coefficient of Determination \\\\\n")
        f.write(f"NRMSE (\\%) & {metrics['NRMSE']:.4f} & Normalized RMSE (RMSE/mean) \\\\\n")
        f.write(f"MAPE (\\%) & {metrics['MAPE']:.4f} & Mean Absolute Percentage Error \\\\\n")
        f.write("\\hline\n")
        f.write("\\end{tabular}\n")
        f.write("\\label{tab:metrics}\n")
        f.write("\\end{table}\n")
        
    print(f"LaTeX metrics table saved to {latex_path}")
    
    return output_path

def analyze_trip_length_distribution(distance_matrix, station_mapping, trips_df, y_true, y_pred, output_dir):
    """
    Analyze trip length distribution for observed and predicted flows
    
    Parameters:
    -----------
    distance_matrix : numpy.ndarray
        Matrix of distances between stations
    station_mapping : dict
        Maps station names to indices
    trips_df : pandas.DataFrame
        DataFrame with trip data
    y_true : array-like
        Ground truth flow values
    y_pred : array-like
        Predicted flow values
    output_dir : str
        Directory to save the analysis results
    """
    print("Analyzing trip length distribution...")
    
    # Calculate observed trip distance distribution from actual data
    trip_distances = []
    
    # Filter trips to include only mapped stations
    mapped_trips = trips_df[
        trips_df['start_station_name'].isin(station_mapping.keys()) & 
        trips_df['end_station_name'].isin(station_mapping.keys())
    ]
    
    # Sample trips if there are too many
    if len(mapped_trips) > 10000:
        mapped_trips = mapped_trips.sample(10000, random_state=42)
    
    # Get distance for each trip
    for _, trip in mapped_trips.iterrows():
        origin_idx = station_mapping.get(trip['start_station_name'])
        dest_idx = station_mapping.get(trip['end_station_name'])
        
        if origin_idx is not None and dest_idx is not None:
            trip_distances.append(distance_matrix[origin_idx, dest_idx])
    
    # Convert to numpy array
    trip_distances = np.array(trip_distances)
    
    # Create bins for distance ranges
    distance_bins = np.linspace(0, max(trip_distances) + 0.5, 20)
    
    # Plot distribution
    plt.figure(figsize=(12, 8))
    
    # Histogram of observed trip distances
    plt.hist(trip_distances, bins=distance_bins, alpha=0.7, color='#3498db', 
             label='Observed Trips', edgecolor='black')
    
    # Add statistics
    mean_dist = np.mean(trip_distances)
    median_dist = np.median(trip_distances)
    
    plt.axvline(mean_dist, color='red', linestyle='--', linewidth=2, 
                label=f'Mean Distance: {mean_dist:.2f} km')
    plt.axvline(median_dist, color='green', linestyle='--', linewidth=2, 
                label=f'Median Distance: {median_dist:.2f} km')
    
    # Customize plot
    plt.title('Trip Distance Distribution', fontsize=16, pad=20)
    plt.xlabel('Trip Distance (km)', fontsize=14, labelpad=10)
    plt.ylabel('Number of Trips', fontsize=14, labelpad=10)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12)
    
    # Save figure
    dist_plot_path = os.path.join(output_dir, 'trip_distance_distribution.png')
    plt.savefig(dist_plot_path, dpi=300, bbox_inches='tight')
    
    plt.close()
    
    # Calculate distance-based metrics
    distance_stats = {
        'mean_distance': mean_dist,
        'median_distance': median_dist,
        'min_distance': np.min(trip_distances),
        'max_distance': np.max(trip_distances),
        'std_distance': np.std(trip_distances)
    }
    
    # Save statistics to file
    stats_path = os.path.join(output_dir, 'distance_statistics.txt')
    with open(stats_path, 'w') as f:
        f.write("Trip Distance Statistics\n")
        f.write("=======================\n")
        f.write(f"Mean Distance: {distance_stats['mean_distance']:.4f} km\n")
        f.write(f"Median Distance: {distance_stats['median_distance']:.4f} km\n")
        f.write(f"Minimum Distance: {distance_stats['min_distance']:.4f} km\n")
        f.write(f"Maximum Distance: {distance_stats['max_distance']:.4f} km\n")
        f.write(f"Standard Deviation: {distance_stats['std_distance']:.4f} km\n")
    
    print(f"Trip distance distribution analysis saved to {dist_plot_path}")
    return distance_stats

def analyze_feature_importance(model, feature_names, output_dir):
    """
    Analyze feature importance for the OD model
    
    Parameters:
    -----------
    model : torch.nn.Module
        Trained model
    feature_names : list
        List of feature names
    output_dir : str
        Directory to save the analysis results
    """
    import torch
    print("Analyzing feature importance...")
    
    # For MLP models, we approximate feature importance using the weights of the first layer
    # This is a simple approach. For more accurate results, permutation importance could be used
    
    # Get the first layer weights
    input_weights = model.mlp[0].weight.data.cpu().numpy()
    
    # Calculate importance as the mean absolute weight for each input feature
    importance = np.abs(input_weights).mean(axis=0)
    
    # Create dictionary of feature importances
    feature_importance = dict(zip(feature_names, importance))
    
    # Sort features by importance
    sorted_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)
    
    # Select top features
    top_n = min(20, len(sorted_features))
    top_features = sorted_features[:top_n]
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    
    feature_names = [f[0] for f in top_features]
    importance_values = [f[1] for f in top_features]
    
    # Create horizontal bar chart
    plt.barh(range(len(feature_names)), importance_values, color='#2ecc71', alpha=0.8)
    plt.yticks(range(len(feature_names)), feature_names)
    
    # Customize plot
    plt.title('Feature Importance Analysis', fontsize=16, pad=20)
    plt.xlabel('Importance (Mean Absolute Weight)', fontsize=14, labelpad=10)
    plt.ylabel('Feature', fontsize=14, labelpad=10)
    plt.grid(True, linestyle='--', alpha=0.7, axis='x')
    
    # Adjust layout
    plt.tight_layout()
    
    # Save figure
    importance_path = os.path.join(output_dir, 'feature_importance.png')
    plt.savefig(importance_path, dpi=300, bbox_inches='tight')
    
    plt.close()
    
    # Save feature importance to file
    importance_txt_path = os.path.join(output_dir, 'feature_importance.txt')
    with open(importance_txt_path, 'w') as f:
        f.write("Feature Importance Analysis\n")
        f.write("==========================\n")
        for name, importance in sorted_features:
            f.write(f"{name:<30} {importance:.6f}\n")
    
    print(f"Feature importance analysis saved to {importance_path}")
    return dict(sorted_features)

def analyze_spatial_error_distribution(station_data, station_mapping, y_true, y_pred, feature_matrix, output_dir):
    """
    Analyze spatial distribution of prediction errors
    
    Parameters:
    -----------
    station_data : dict
        Dictionary with station metadata
    station_mapping : dict
        Maps station names to indices
    y_true : array-like
        Ground truth flow values
    y_pred : array-like
        Predicted flow values
    feature_matrix : numpy.ndarray
        Matrix of features used for prediction
    output_dir : str
        Directory to save the analysis results
    """
    print("Analyzing spatial error distribution...")
    
    # Calculate errors
    errors = y_pred - y_true
    abs_errors = np.abs(errors)
    rel_errors = np.abs(errors) / (y_true + 1e-8)  # Add small epsilon to avoid division by zero
    
    # Create a DataFrame for station errors
    station_errors = {
        'station_id': [],
        'station_name': [],
        'lat': [],
        'lng': [],
        'total_outflow': [],
        'total_inflow': [],
        'mean_error': [],
        'mean_abs_error': [],
        'mean_rel_error': []
    }
    
    # Extract origin and destination indices from feature matrix
    # This depends on how your features are structured
    # For this example, let's assume we can identify origin and destination from features
    
    # Group errors by station (simplified approach)
    for station_idx in range(len(station_data)):
        station_name = station_data[station_idx]['name']
        
        # Get all errors where this station is origin or destination
        # This is a simplification - you'll need to adapt to your feature structure
        origin_mask = feature_matrix[:, 3] == 1  # Is origin indicator
        dest_mask = feature_matrix[:, 4] == 1    # Is destination indicator
        
        # Sum flows where station is origin (outflow)
        outflow = np.sum(y_true[origin_mask])
        
        # Sum flows where station is destination (inflow)
        inflow = np.sum(y_true[dest_mask])
        
        # Calculate mean errors
        station_errors['station_id'].append(station_idx)
        station_errors['station_name'].append(station_name)
        station_errors['lat'].append(station_data[station_idx]['lat'])
        station_errors['lng'].append(station_data[station_idx]['lng'])
        station_errors['total_outflow'].append(outflow)
        station_errors['total_inflow'].append(inflow)
        station_errors['mean_error'].append(np.mean(errors[origin_mask | dest_mask]))
        station_errors['mean_abs_error'].append(np.mean(abs_errors[origin_mask | dest_mask]))
        station_errors['mean_rel_error'].append(np.mean(rel_errors[origin_mask | dest_mask]))
    
    # Convert to DataFrame
    error_df = pd.DataFrame(station_errors)
    
    # Plot error map
    plt.figure(figsize=(14, 10))
    
    # Set up the figure
    ax = plt.gca()
    ax.set_facecolor('#f8f8f8')
    ax.set_aspect('equal')
    plt.grid(False)
    
    # Create custom colormap for errors
    cmap = LinearSegmentedColormap.from_list(
        'error_cmap', ['#2ecc71', '#f1c40f', '#e74c3c'], N=256
    )
    
    # Size based on total flow
    total_flow = error_df['total_inflow'] + error_df['total_outflow']
    sizes = 100 + 1000 * (total_flow / total_flow.max())
    
    # Color based on mean absolute error
    norm_errors = error_df['mean_abs_error'] / error_df['mean_abs_error'].max()
    
    # Scatter plot of stations with errors
    scatter = plt.scatter(
        error_df['lng'],
        error_df['lat'],
        s=sizes,
        c=norm_errors,
        cmap=cmap,
        alpha=0.7,
        edgecolors='black',
        linewidths=0.5
    )
    
    # Add colorbar
    cbar = plt.colorbar(scatter, shrink=0.7)
    cbar.set_label('Normalized Mean Absolute Error', fontsize=12)
    
    # Add labels for top error stations
    top_error_stations = error_df.nlargest(5, 'mean_abs_error')
    for _, station in top_error_stations.iterrows():
        plt.annotate(
            station['station_name'],
            (station['lng'], station['lat']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8)
        )
    
    # Set axis limits
    lngs = error_df['lng']
    lats = error_df['lat']
    lng_pad = (lngs.max() - lngs.min()) * 0.1
    lat_pad = (lats.max() - lats.min()) * 0.1
    plt.xlim(lngs.min() - lng_pad, lngs.max() + lng_pad)
    plt.ylim(lats.min() - lat_pad, lats.max() + lat_pad)
    
    # Title and labels
    plt.title('Spatial Distribution of Prediction Errors', fontsize=16, pad=20)
    plt.xlabel('Longitude', fontsize=14, labelpad=10)
    plt.ylabel('Latitude', fontsize=14, labelpad=10)
    
    # Remove tick labels for cleaner map
    plt.xticks([])
    plt.yticks([])
    
    # Save figure
    error_map_path = os.path.join(output_dir, 'spatial_error_distribution.png')
    plt.savefig(error_map_path, dpi=300, bbox_inches='tight')
    
    plt.close()
    
    # Save station error data
    error_data_path = os.path.join(output_dir, 'station_errors.csv')
    error_df.to_csv(error_data_path, index=False)
    
    print(f"Spatial error distribution analysis saved to {error_map_path}")
    return error_df

def create_comprehensive_report(metrics, output_dir, filename='od_evaluation_report.txt'):
    """
    Create a comprehensive report of all metrics and analyses
    
    Parameters:
    -----------
    metrics : dict
        Dictionary of all metrics and statistics
    output_dir : str
        Directory to save the report
    filename : str
        Filename for the report
    """
    report_path = os.path.join(output_dir, filename)
    
    with open(report_path, 'w') as f:
        f.write("==============================================================================\n")
        f.write("                  ORIGIN-DESTINATION MODEL EVALUATION REPORT                  \n")
        f.write("==============================================================================\n\n")
        
        # Basic error metrics
        f.write("1. ERROR METRICS\n")
        f.write("----------------\n")
        f.write(f"Mean Absolute Error (MAE): {metrics.get('MAE', 'N/A'):.4f}\n")
        f.write(f"Mean Squared Error (MSE): {metrics.get('MSE', 'N/A'):.4f}\n")
        f.write(f"Root Mean Squared Error (RMSE): {metrics.get('RMSE', 'N/A'):.4f}\n")
        f.write(f"Normalized RMSE: {metrics.get('NRMSE', 'N/A'):.4f}\n")
        f.write(f"Mean Absolute Percentage Error (MAPE): {metrics.get('MAPE', 'N/A'):.4f}%\n")
        f.write(f"RÂ² Score: {metrics.get('R2', 'N/A'):.4f}\n\n")
        
        # Trip distance statistics
        if 'distance_stats' in metrics:
            dist_stats = metrics['distance_stats']
            f.write("2. TRIP DISTANCE STATISTICS\n")
            f.write("---------------------------\n")
            f.write(f"Mean Trip Distance: {dist_stats.get('mean_distance', 'N/A'):.4f} km\n")
            f.write(f"Median Trip Distance: {dist_stats.get('median_distance', 'N/A'):.4f} km\n")
            f.write(f"Minimum Trip Distance: {dist_stats.get('min_distance', 'N/A'):.4f} km\n")
            f.write(f"Maximum Trip Distance: {dist_stats.get('max_distance', 'N/A'):.4f} km\n")
            f.write(f"Standard Deviation: {dist_stats.get('std_distance', 'N/A'):.4f} km\n\n")
        
        # OD matrix statistics
        if 'od_stats' in metrics:
            od_stats = metrics['od_stats']
            f.write("3. OD MATRIX STATISTICS\n")
            f.write("-----------------------\n")
            f.write(f"Total Flow Volume: {od_stats.get('total_flow', 'N/A'):.0f}\n")
            f.write(f"Zero-Flow Percentage: {od_stats.get('zero_flow_pct', 'N/A'):.2f}%\n")
            f.write(f"Mean Flow: {od_stats.get('mean_flow', 'N/A'):.4f}\n")
            f.write(f"Median Flow: {od_stats.get('median_flow', 'N/A'):.4f}\n")
            f.write(f"Maximum Flow: {od_stats.get('max_flow', 'N/A'):.4f}\n\n")
        
        # Temporal analysis
        if 'temporal_stats' in metrics:
            temp_stats = metrics['temporal_stats']
            f.write("4. TEMPORAL ANALYSIS\n")
            f.write("--------------------\n")
            f.write(f"Morning Peak Flow: {temp_stats.get('morning_peak_flow', 'N/A'):.0f}\n")
            f.write(f"Evening Peak Flow: {temp_stats.get('evening_peak_flow', 'N/A'):.0f}\n")
            f.write(f"Weekday/Weekend Flow Ratio: {temp_stats.get('weekday_weekend_ratio', 'N/A'):.2f}\n\n")
        
        # Spatial analysis
        if 'spatial_stats' in metrics:
            spatial_stats = metrics['spatial_stats']
            f.write("5. SPATIAL ANALYSIS\n")
            f.write("-------------------\n")
            f.write(f"Number of Stations: {spatial_stats.get('num_stations', 'N/A')}\n")
            f.write(f"Average Station Connections: {spatial_stats.get('avg_connections', 'N/A'):.2f}\n")
            f.write(f"Most Popular Origin: {spatial_stats.get('top_origin', 'N/A')}\n")
            f.write(f"Most Popular Destination: {spatial_stats.get('top_destination', 'N/A')}\n\n")
        
        f.write("==============================================================================\n")
        f.write("                                END OF REPORT                                 \n")
        f.write("==============================================================================\n")
    
    print(f"Comprehensive evaluation report saved to {report_path}")
    return report_path

def evaluate_od_model(model, X, y, station_data, station_mapping, distance_matrix, trips_df, feature_names, output_dir):
    """
    Comprehensive evaluation of an Origin-Destination model with all recommended metrics
    
    Parameters:
    -----------
    model : object
        Trained model with predict method
    X : numpy.ndarray
        Feature matrix
    y : numpy.ndarray
        Ground truth target values
    station_data : dict
        Dictionary with station metadata
    station_mapping : dict
        Maps station names to indices
    distance_matrix : numpy.ndarray
        Matrix of distances between stations
    trips_df : pandas.DataFrame
        DataFrame with trip data
    feature_names : list
        List of feature names
    output_dir : str
        Directory to save evaluation results
        
    Returns:
    --------
    metrics : dict
        Dictionary of all evaluation metrics and analyses
    """
    # Create evaluation directory
    eval_dir = os.path.join(output_dir, 'evaluation')
    os.makedirs(eval_dir, exist_ok=True)
    
    print("Performing comprehensive model evaluation...")
    
    # Get predictions
    y_pred = model.predict(X)
    
    # Calculate standard and additional metrics
    metrics = calculate_additional_metrics(y, y_pred)
    
    # Create metrics table
    create_metrics_table(metrics, eval_dir)
    
    # Analyze trip length distribution
    distance_stats = analyze_trip_length_distribution(
        distance_matrix, station_mapping, trips_df, y, y_pred, eval_dir
    )
    metrics['distance_stats'] = distance_stats
    
    # Analyze feature importance
    feature_importance = analyze_feature_importance(
        model, feature_names, eval_dir
    )
    metrics['feature_importance'] = feature_importance
    
    # Analyze spatial error distribution
    error_df = analyze_spatial_error_distribution(
        station_data, station_mapping, y, y_pred, X, eval_dir
    )
    metrics['error_df'] = error_df
    
    # Create comprehensive report
    create_comprehensive_report(metrics, eval_dir)
    
    print("Comprehensive evaluation completed and saved to:", eval_dir)
    return metrics

def extended_evaluate_model(model, X, y, station_data, station_mapping, distance_matrix, trips_df, config, device='cuda'):
    """Extended evaluation with additional metrics"""
    # Get predictions
    predictions = predict(model, X, device)
    
    # Calculate additional metrics
    additional_metrics = calculate_additional_metrics(y, predictions)
    
    # Create metrics table
    metrics_path = create_metrics_table(additional_metrics, config['output_path'])
    print(f"Metrics table saved to {metrics_path}")
    
    # Create feature names list (must match your actual feature structure)
    feature_names = []
    
    # Origin features (5)
    feature_names.extend(['origin_lat', 'origin_lng', 'origin_popularity', 'is_origin', 'is_dest'])
    
    # Destination features (5)
    feature_names.extend(['dest_lat', 'dest_lng', 'dest_popularity', 'is_not_origin', 'is_not_dest'])
    
    # Top 5 nearby stations features (20)
    for i in range(5):
        feature_names.extend([f'nearby{i+1}_lat', f'nearby{i+1}_lng', f'nearby{i+1}_popularity', f'nearby{i+1}_distance'])
    
    # Spatial features (3)
    feature_names.extend(['od_distance', 'lat_difference', 'lng_difference'])
    
    # Temporal features (13)
    feature_names.extend([
        'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos',
        'is_morning_peak', 'is_evening_peak', 'is_weekend',
        'is_winter', 'is_spring', 'is_summer', 'is_fall'
    ])
    
    # Analyze trip length distribution
    analyze_trip_length_distribution(
        distance_matrix, station_mapping, trips_df, y, predictions, config['output_path']
    )
    
    # Analyze feature importance
    analyze_feature_importance(model, feature_names, config['output_path'])
    
    # Analyze spatial error distribution
    analyze_spatial_error_distribution(
        station_data, station_mapping, y, predictions, X, config['output_path']
    )
    
    return additional_metrics

###########################################
# Visualization Functions
###########################################

def plot_training_history(history, output_dir):
    """Plot training and validation metrics over epochs with improved styling"""
    print("Plotting training history...")
    
    # Calculate RMSE from MSE values
    train_rmse = [np.sqrt(mse) for mse in history['train_loss']]
    val_rmse = [np.sqrt(mse) for mse in history['val_loss']]
    
    # Set global font sizes
    plt.rcParams.update({
        'font.size': 16,           # Increase base font size
        'axes.titlesize': 20,      # Title font size
        'axes.labelsize': 18,      # Axis label font size
        'xtick.labelsize': 16,     # X tick label size
        'ytick.labelsize': 16,     # Y tick label size
        'legend.fontsize': 16,     # Legend font size
        'figure.titlesize': 22     # Figure title size
    })
    
    # Create figure with two subplots
    plt.figure(figsize=(14, 10))
    
    # Plot training and validation RMSE
    plt.subplot(2, 1, 1)
    plt.plot(train_rmse, marker='o', markersize=8, linestyle='-', linewidth=2, label='Training RMSE')
    plt.plot(val_rmse, marker='s', markersize=8, linestyle='-', linewidth=2, label='Validation RMSE')
    plt.title('Training and Validation Root Mean Square Error', pad=15)
    plt.xlabel('Epoch', labelpad=10)
    plt.ylabel('RMSE', labelpad=10)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)
    
    # Plot training and validation MAE
    plt.subplot(2, 1, 2)
    plt.plot(history['train_mae'], marker='o', markersize=8, linestyle='-', linewidth=2, label='Training MAE')
    plt.plot(history['val_mae'], marker='s', markersize=8, linestyle='-', linewidth=2, label='Validation MAE')
    plt.title('Training and Validation Mean Absolute Error', pad=15)
    plt.xlabel('Epoch', labelpad=10)
    plt.ylabel('Mean Absolute Error (MAE)', labelpad=10)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)
    
    plt.tight_layout(pad=3.0)
    
    # Save plots
    plot_path_png = os.path.join(output_dir, 'training_history.png')
    plt.savefig(plot_path_png, dpi=150)
    
    plot_path_svg = os.path.join(output_dir, 'training_history.svg')
    plt.savefig(plot_path_svg, format='svg')
    
    plt.close()
    
    print(f"Training history plots saved to {plot_path_png} and {plot_path_svg}")

def visualize_predictions(model, X, y, output_dir, num_samples=100, device='cuda'):
    """Visualize model predictions vs. ground truth"""
    print("Visualizing predictions...")
    
    # Set global font sizes
    plt.rcParams.update({
        'font.size': 16,
        'axes.titlesize': 20,
        'axes.labelsize': 18,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16,
        'legend.fontsize': 16,
        'figure.titlesize': 22
    })
    
    # Make predictions
    predictions = predict(model, X, device)
    
    # Limit to num_samples
    if len(y) > num_samples:
        indices = np.random.choice(len(y), num_samples, replace=False)
        y_subset = y[indices]
        predictions_subset = predictions[indices]
    else:
        y_subset = y
        predictions_subset = predictions
    
    # Create scatter plot
    plt.figure(figsize=(12, 10))
    
    # Plot points with improved styling
    plt.scatter(y_subset, predictions_subset, alpha=0.6, s=80, c='#3498db', edgecolors='#2980b9')
    
    # Add perfect prediction line
    max_val = max(np.max(y_subset), np.max(predictions_subset))
    plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2.5, label='Perfect Prediction')
    
    # Calculate and display RÂ² value
    r2 = r2_score(y_subset, predictions_subset)
    plt.text(0.05, 0.95, f'RÂ² = {r2:.4f}', transform=plt.gca().transAxes, 
             fontsize=16, verticalalignment='top',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))
    
    # Add labels and title
    plt.title('Model Predictions vs. Ground Truth', pad=15)
    plt.xlabel('Ground Truth', labelpad=10)
    plt.ylabel('Predictions', labelpad=10)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(frameon=True, fancybox=True, shadow=True)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(output_dir, 'prediction_scatter.png')
    plt.savefig(plot_path, dpi=150)
    
    plot_path_svg = os.path.join(output_dir, 'prediction_scatter.svg')
    plt.savefig(plot_path_svg, format='svg')
    
    plt.close()
    
    # Create histogram of errors
    errors = predictions - y
    
    plt.figure(figsize=(12, 8))
    
    # Create histogram
    plt.hist(errors, bins=50, alpha=0.75, color='#2ecc71', edgecolor='#27ae60')
    
    # Add a vertical line at zero error
    plt.axvline(x=0, color='#e74c3c', linestyle='--', linewidth=2.5, label='Zero Error')
    
    # Add distribution statistics
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    
    stats_text = f'Mean Error: {mean_error:.4f}\nStd Dev: {std_error:.4f}'
    plt.text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, 
             fontsize=16, verticalalignment='top',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))
    
    # Add labels and title
    plt.title('Distribution of Prediction Errors', pad=15)
    plt.xlabel('Prediction Error', labelpad=10)
    plt.ylabel('Frequency', labelpad=10)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(frameon=True, fancybox=True, shadow=True)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save plot
    error_plot_path = os.path.join(output_dir, 'error_distribution.png')
    plt.savefig(error_plot_path, dpi=150)
    
    error_plot_path_svg = os.path.join(output_dir, 'error_distribution.svg')
    plt.savefig(error_plot_path_svg, format='svg')
    
    plt.close()
    
    print(f"Prediction visualization saved to {plot_path} and {error_plot_path}")


def create_od_matrix_heatmap(od_matrix, period_name, output_dir, color_palette='YlOrRd', description=None):
    """
    Create and save OD matrix heatmap visualization
    
    Parameters:
    -----------
    od_matrix : pandas.DataFrame
        Origin-Destination matrix
    period_name : str
        Name of the time period (e.g., 'Morning Peak')
    output_dir : str
        Directory to save the heatmap
    color_palette : str
        Color palette to use for the heatmap
    description : str, optional
        Description text to add to the figure
    """
    plt.figure(figsize=(14, 12))
    
    # Create heatmap
    ax = sns.heatmap(
        od_matrix,
        cmap=color_palette,
        square=True,
        cbar_kws={'label': 'Number of Trips'},
        linewidths=0.1,
        linecolor='white'
    )
    
    # Customize plot
    plt.title(f'Origin-Destination Matrix: {period_name}', fontsize=16, pad=20)
    plt.xlabel('Destination Station', fontsize=14, labelpad=10)
    plt.ylabel('Origin Station', fontsize=14, labelpad=10)
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    
    # Add description text box if provided
    if description:
        plt.figtext(
            0.5, 0.01,
            description,
            ha='center',
            fontsize=12,
            bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.5')
        )
    
    # Create safe filename
    safe_name = period_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    
    # Save as PNG
    png_path = os.path.join(output_dir, f'{safe_name}_od_matrix_heatmap.png')
    plt.savefig(png_path, dpi=300, bbox_inches='tight')
    
    # Save as SVG
    svg_path = os.path.join(output_dir, f'{safe_name}_od_matrix_heatmap.svg')
    plt.savefig(svg_path, format='svg', bbox_inches='tight')
    
    print(f" OD matrix heatmap saved as PNG: {png_path}")
    print(f" OD matrix heatmap saved as SVG: {svg_path}")
    
    plt.close()
    
    return png_path, svg_path

def analyze_seasonal_patterns(trips_df, output_dir, config):
    """Analyze seasonal patterns in Citibike trip data"""
    print("Analyzing seasonal patterns...")
    
    # Create output directory
    seasonal_dir = os.path.join(output_dir, 'seasonal_analysis')
    os.makedirs(seasonal_dir, exist_ok=True)
    
    # Define seasons
    seasons = config.get('seasons', {
        'winter': [1, 2, 12],    # Jan, Feb, Dec
        'spring': [3, 4, 5],     # Mar, Apr, May
        'summer': [6, 7, 8],     # Jun, Jul, Aug
        'fall': [9, 10, 11]      # Sep, Oct, Nov
    })
    
    # Check if we have month data
    if 'month' not in trips_df.columns:
        print("No month data available in trip data, extracting from datetime")
        if 'started_at' in trips_df.columns:
            trips_df['month'] = pd.to_datetime(trips_df['started_at']).dt.month
        else:
            print("Cannot analyze seasonal patterns: no month or datetime data available")
            return seasonal_dir
    
    # Get available months
    available_months = trips_df['month'].unique()
    print(f"Available months: {sorted(available_months)}")
    
    # Group trips by month
    monthly_trips = trips_df.groupby('month').size()
    
    # Calculate trips by season
    seasonal_trips = {}
    for season_name, season_months in seasons.items():
        # Filter to months we have data for
        valid_months = [m for m in season_months if m in available_months]
        if valid_months:
            seasonal_trips[season_name] = monthly_trips[valid_months].sum()
    
    # Create seasonal trip count visualization
    plt.figure(figsize=(12, 8))
    
    # Set color palette for seasons
    season_colors = {
        'winter': '#A1D6E2',  # Light blue
        'spring': '#81C784',  # Green
        'summer': '#FFB74D',  # Orange
        'fall': '#E57373'     # Red
    }
    
    # Plot seasonal trip counts
    bars = plt.bar(
        range(len(seasonal_trips)),
        list(seasonal_trips.values()),
        width=0.7,
        color=[season_colors.get(s, '#BBBBBB') for s in seasonal_trips.keys()]
    )
    
    # Add trip count labels on the bars
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2.,
            height * 1.02,
            f'{int(height):,}',
            ha='center',
            va='bottom',
            fontsize=12
        )
    
    # Customize plot
    plt.title('Citibike Trips by Season', fontsize=16, pad=20)
    plt.xlabel('Season', fontsize=14, labelpad=10)
    plt.ylabel('Number of Trips', fontsize=14, labelpad=10)
    plt.xticks(range(len(seasonal_trips)), [s.capitalize() for s in seasonal_trips.keys()], fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Save figure
    seasonal_counts_path = os.path.join(seasonal_dir, 'seasonal_trip_counts.png')
    plt.savefig(seasonal_counts_path, dpi=300, bbox_inches='tight')
    
    plt.close()
    
    # Additional seasonal analysis code can go here (monthly patterns, trip durations, etc.)
    # See original code for details

    return seasonal_dir

def analyze_day_type_patterns(trips_df, output_dir, config):
    """Analyze weekday vs weekend patterns in Citibike trip data"""
    print("Analyzing weekday vs weekend patterns...")
    
    # Create output directory
    day_type_dir = os.path.join(output_dir, 'day_type_analysis')
    os.makedirs(day_type_dir, exist_ok=True)
    
    # Check if we have day type data
    if 'is_weekend' not in trips_df.columns and 'start_day' not in trips_df.columns:
        print("No day type data available in trip data, extracting from datetime")
        if 'started_at' in trips_df.columns:
            trips_df['start_day'] = pd.to_datetime(trips_df['started_at']).dt.dayofweek
            trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
        else:
            print("Cannot analyze day type patterns: no day or datetime data available")
            return day_type_dir
    
    # Ensure is_weekend is available
    if 'is_weekend' not in trips_df.columns and 'start_day' in trips_df.columns:
        trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Create color scheme
    day_type_colors = {
        'weekday': '#3498db',  # Blue for weekdays
        'weekend': '#e74c3c'   # Red for weekends
    }
    
    # Count trips by day type
    day_type_counts = trips_df.groupby('is_weekend').size()
    day_type_labels = ['Weekday', 'Weekend']
    
    # Create trip count visualization
    plt.figure(figsize=(10, 8))
    
    # Plot day type counts
    bars = plt.bar(
        day_type_labels,
        [day_type_counts.get(0, 0), day_type_counts.get(1, 0)],
        width=0.7,
        color=[day_type_colors['weekday'], day_type_colors['weekend']]
    )
    
    # Add count labels
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2.,
            height * 1.02,
            f'{int(height):,}',
            ha='center',
            va='bottom',
            fontsize=14
        )
    
    # Customize plot
    plt.title('Citibike Trips: Weekday vs Weekend', fontsize=16, pad=20)
    plt.ylabel('Number of Trips', fontsize=14, labelpad=10)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(fontsize=14)
    
    # Save figure
    day_type_counts_path = os.path.join(day_type_dir, 'day_type_trip_counts.png')
    plt.savefig(day_type_counts_path, dpi=300, bbox_inches='tight')
    
    plt.close()
    
    # Additional day type analysis code can go here (hourly patterns, top stations, etc.)
    # See original code for details
    
    return day_type_dir

def generate_flow_maps(trips_df, output_dir, config, top_stations=None, 
                      show_top_flows=True, top_flows_count=50, 
                      min_flow_threshold=10):
    """Enhanced flow map generator with improved visual segmentation of flows"""
    print("Generating flow maps and OD matrix heatmaps...")
    
    # Create output directories
    flow_dir = os.path.join(output_dir, 'flow_visualizations')
    heatmap_dir = os.path.join(output_dir, 'od_matrix_heatmaps')
    os.makedirs(flow_dir, exist_ok=True)
    os.makedirs(heatmap_dir, exist_ok=True)
    
    # Define time periods with custom colors
    time_periods = {
        'morning_peak': {
            'name': 'Morning Peak (7-10 AM)',
            'start_hour': config['peak_morning_start'],
            'end_hour': config['peak_morning_end'],
            'description': 'Strong flows from residential to commercial zones',
            'color_palette': 'Blues',
            'base_color': '#0d47a1'  # Dark blue for morning peak (much darker than original)
        },
        'afternoon': {
            'name': 'Afternoon (10 AM-4 PM)',
            'start_hour': config['peak_morning_end'],
            'end_hour': config['peak_evening_start'],
            'description': 'Balanced flows with commercial-commercial bias',
            'color_palette': 'Greens',
            'base_color': '#004d40'  # Dark green
        },
        'evening_peak': {
            'name': 'Evening Peak (4-7 PM)',
            'start_hour': config['peak_evening_start'],
            'end_hour': config['peak_evening_end'],
            'description': 'Strong reverse flows from commercial to residential zones',
            'color_palette': 'Purples',
            'base_color': '#4a148c'  # Dark purple
        },
        'all_day': {
            'name': 'All Day',
            'start_hour': 0,
            'end_hour': 24,
            'description': 'Combined patterns showing overall mobility structure',
            'color_palette': 'viridis',
            'base_color': '#1a237e'  # Dark blue-violet
        }
    }
    
    # Process each time period
    for period_key, period_info in time_periods.items():
        print(f"Generating visualization for {period_info['name']}")
        
        # Filter trips for this period
        period_trips = trips_df[
            (trips_df['start_hour'] >= period_info['start_hour']) & 
            (trips_df['start_hour'] < period_info['end_hour'])
        ]
        
        # Only include top stations
        period_trips = period_trips[
            period_trips['start_station_name'].isin(top_stations['station_name']) & 
            period_trips['end_station_name'].isin(top_stations['station_name'])
        ]
        
        print(f" Number of trips in this period: {len(period_trips)}")
        
        # Create OD matrix
        od_matrix = pd.crosstab(
            period_trips['start_station_name'], 
            period_trips['end_station_name'],
            dropna=False
        ).fillna(0)
        
        print(f" OD matrix shape: {od_matrix.shape}")
        
        # Create flow map with improved visualization
        plt.figure(figsize=(16, 14), dpi=300)
        ax = plt.gca()
        ax.set_facecolor('#f8f8f8')
        
        # Calculate all flows between stations
        flows = []
        for origin in od_matrix.index:
            for destination in od_matrix.columns:
                if origin != destination:
                    flow_count = od_matrix.loc[origin, destination]
                    if flow_count >= min_flow_threshold:
                        flows.append((origin, destination, flow_count))
        
        # Sort flows by count and filter to top flows if requested
        flows.sort(key=lambda x: x[2], reverse=True)
        if show_top_flows and len(flows) > top_flows_count:
            flows = flows[:top_flows_count]
        
        # Get flow range for scaling
        if flows:
            flow_values = [f[2] for f in flows]
            max_flow = max(flow_values)
            min_flow = min(flow_values)
            
            # Quartiles for better visual segmentation
            q1 = min_flow + (max_flow - min_flow) * 0.25
            q2 = min_flow + (max_flow - min_flow) * 0.5
            q3 = min_flow + (max_flow - min_flow) * 0.75
            
            # Plot stations as nodes
            for _, station in top_stations.iterrows():
                plt.scatter(
                    station['lng'], 
                    station['lat'],
                    s=100,  # Larger fixed size for better visibility
                    c='white',
                    edgecolors='black',
                    linewidth=0.8,
                    zorder=3
                )
            
            # Create color maps for each quartile
            base_color = matplotlib.colors.to_rgb(period_info['base_color'])
            colors = {
                'q1': tuple([c * 0.8 for c in base_color]),  # Lightest 
                'q2': tuple([c * 0.6 for c in base_color]),  # Medium light
                'q3': tuple([c * 0.4 for c in base_color]),  # Medium dark
                'q4': tuple([c * 0.2 for c in base_color])   # Darkest
            }
            
            # Line width for each quartile (much more dramatic differences)
            widths = {
                'q1': 0.5,   # Thinnest for low volume
                'q2': 1.5,   # Medium thin
                'q3': 3.0,   # Medium thick
                'q4': 5.0    # Thickest for high volume
            }
            
            # Draw flow lines with improved styling
            for origin, destination, flow in flows:
                # Get station coordinates
                origin_station = top_stations[top_stations['station_name'] == origin].iloc[0]
                dest_station = top_stations[top_stations['station_name'] == destination].iloc[0]
                
                # Calculate bezier curve control point
                start_x, start_y = origin_station['lng'], origin_station['lat']
                end_x, end_y = dest_station['lng'], dest_station['lat']
                
                # Calculate perpendicular vector for curve
                dx = end_x - start_x
                dy = end_y - start_y
                length = np.sqrt(dx*dx + dy*dy)
                
                # Adjust curvature based on distance
                curve_strength = min(0.3, 0.05 + 0.1 * (length / 0.05))
                
                # Midpoint and control point
                mid_x = (start_x + end_x) / 2
                mid_y = (start_y + end_y) / 2
                control_x = mid_x - dy * curve_strength
                control_y = mid_y + dx * curve_strength
                
                # Determine which quartile this flow belongs to
                if flow <= q1:
                    color = colors['q1']
                    width = widths['q1']
                    alpha = 0.4
                elif flow <= q2:
                    color = colors['q2']
                    width = widths['q2']
                    alpha = 0.6
                elif flow <= q3:
                    color = colors['q3']
                    width = widths['q3']
                    alpha = 0.8
                else:
                    color = colors['q4']
                    width = widths['q4']
                    alpha = 1.0
                
                # Generate curved line points
                t = np.linspace(0, 1, 50)
                bezier_x = (1-t)**2 * start_x + 2*(1-t)*t * control_x + t**2 * end_x
                bezier_y = (1-t)**2 * start_y + 2*(1-t)*t * control_y + t**2 * end_y
                
                # Plot the curved line
                plt.plot(
                    bezier_x, bezier_y,
                    color=color,
                    linewidth=width,
                    alpha=alpha,
                    solid_capstyle='round',
                    zorder=2
                )
                
                # Add arrow to major flows
                if flow > q3:  # Only add arrows to significant flows
                    arrow_idx = int(len(bezier_x) * 0.7)
                    plt.annotate(
                        '',
                        xy=(bezier_x[arrow_idx], bezier_y[arrow_idx]),
                        xytext=(bezier_x[arrow_idx-3], bezier_y[arrow_idx-3]),
                        arrowprops=dict(
                            arrowstyle="->",
                            color=color,
                            linewidth=width*0.8
                        ),
                        zorder=4
                    )
            
            # Add legend with flow levels
            custom_lines = [
                matplotlib.lines.Line2D([0], [0], color=colors['q1'], lw=widths['q1'], alpha=0.4),
                matplotlib.lines.Line2D([0], [0], color=colors['q2'], lw=widths['q2'], alpha=0.6),
                matplotlib.lines.Line2D([0], [0], color=colors['q3'], lw=widths['q3'], alpha=0.8),
                matplotlib.lines.Line2D([0], [0], color=colors['q4'], lw=widths['q4'], alpha=1.0)
            ]
            
            legend_labels = [
                f"< {int(q1)} trips",
                f"{int(q1)}-{int(q2)} trips",
                f"{int(q2)}-{int(q3)} trips",
                f"> {int(q3)} trips"
            ]
            
            plt.legend(
                custom_lines, legend_labels,
                title="Flow Volume",
                loc="upper right",
                frameon=True,
                facecolor='white',
                edgecolor='gray'
            )
        
        # Add title and finalize
        plt.title(f'Flow Map: {period_info["name"]}', fontsize=16, pad=20)
        plt.axis('off')
        plt.tight_layout()
        
        # Save as high-resolution image
        flow_map_path = os.path.join(flow_dir, f'{period_key}_flow_map.png')
        plt.savefig(flow_map_path, dpi=300, bbox_inches='tight')
        print(f" Flow map saved to {flow_map_path}")
        
        # Save vector version for publications
        svg_flow_path = os.path.join(flow_dir, f'{period_key}_flow_map.svg')
        plt.savefig(svg_flow_path, format='svg')
        print(f" Flow map saved as SVG: {svg_flow_path}")
        
        plt.close()
        
        # Create OD Matrix heatmap (keeping the original implementation)
        plt.figure(figsize=(14, 12))
        ax = sns.heatmap(
            od_matrix,
            cmap=period_info['color_palette'],
            square=True,
            cbar_kws={'label': 'Number of Trips'},
            linewidths=0.1,
            linecolor='white'
        )
        
        # Rest of the OD matrix heatmap code...
        plt.title(f'Origin-Destination Matrix: {period_info["name"]}', fontsize=16, pad=20)
        plt.xlabel('Destination Station', fontsize=14, labelpad=10)
        plt.ylabel('Origin Station', fontsize=14, labelpad=10)
        plt.xticks(rotation=90, fontsize=8)
        plt.yticks(rotation=0, fontsize=8)
        
        # Save heatmap
        heatmap_path = os.path.join(heatmap_dir, f'{period_key}_od_matrix_heatmap.png')
        plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
        svg_path = os.path.join(heatmap_dir, f'{period_key}_od_matrix_heatmap.svg')
        plt.savefig(svg_path, format='svg')
        plt.close()
    
    print(f"Flow maps saved to {flow_dir}")
    print(f"OD matrix heatmaps saved to {heatmap_dir}")
    
    return flow_dir, heatmap_dir


###########################################
# Main Execution Functions
###########################################

def process_citibike_data(data_files_pattern, config=None, mode='all'):
    """Process Citibike data and run analysis"""
    if config is None:
        config = CITIBIKE_CONFIG
    
    # Create output directory
    os.makedirs(config['output_path'], exist_ok=True)
    
    # Set device
    device = 'cuda' if torch.cuda.is_available() and RUNTIME_CONFIG['USE_GPU'] else 'cpu'
    print(f"Using device: {device}")
    
    # Load data
    trips_df = load_citibike_data(
        data_files_pattern, 
        sample_rate=RUNTIME_CONFIG['SAMPLE_RATE']
    )
    
    # Get top stations and create station mapping
    station_mapping, station_data, top_station_names = get_top_stations(
        trips_df, config['num_stations']
    )
    
    # Update config with actual number of stations
    config['num_stations'] = len(station_mapping)
    
    # Calculate distance matrix
    distance_matrix = calculate_distance_matrix(station_data)
    
    # Create adjacency matrix for GCN
    adjacency_matrix = prepare_adjacency_matrix(
        distance_matrix, 
        threshold=RUNTIME_CONFIG['DISTANCE_THRESHOLD']
    )
    
    # Run visualizations if requested
    if mode in ['visualize', 'enhanced-viz', 'all']:
        print("Generating visualizations...")
        
        if config.get('analyze_seasons', True):
            analyze_seasonal_patterns(trips_df, config['output_path'], config)
        
        if config.get('analyze_day_types', True):
            analyze_day_type_patterns(trips_df, config['output_path'], config)
        
        top_stations_df = pd.DataFrame([
            {'station_name': station_data[i]['name'], 
             'lat': station_data[i]['lat'], 
             'lng': station_data[i]['lng'],
             'total_trips': station_data[i]['popularity']} 
            for i in range(len(station_data))
        ])
        
        flow_dir, heatmap_dir = generate_flow_maps(
            trips_df,
            config['output_path'],
            config,
            top_stations=top_stations_df,
            show_top_flows=False,
            top_flows_count=30
        )
        print(f"Flow maps saved to {flow_dir}")
        print(f"OD matrix heatmaps saved to {heatmap_dir}")
    
    # Run model training if requested
    if mode in ['train', 'all']:
        print("Preparing data for model training...")
        
        od_matrices = create_od_matrices(trips_df, station_mapping, config['num_stations'])
        X, y = create_features_and_targets(od_matrices, station_data, distance_matrix, config)
        
        scaler_X = StandardScaler()
        X_scaled = scaler_X.fit_transform(X)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.2, random_state=42
        )
        
        print(f"Created {len(X)} feature vectors with dimension {X.shape[1]}")
        print(f"Training set: {len(X_train)} samples, Test set: {len(X_test)} samples")
        
        model = CitibikeODModel(config, X_scaled.shape[1], adjacency_matrix)
        model.to(device)
        
        model, history = train_model(
            model, X_train, y_train, X_test, y_test, config, device
        )
        
        plot_training_history(history, config['output_path'])
        
        print("Evaluating model on test data...")
        metrics = evaluate_model(model, X_test, y_test, device)
        
        print("Running extended model evaluation...")
        feature_names = create_feature_names()
        
        additional_metrics = extended_evaluate_model(
            model, X_test, y_test, station_data, station_mapping, 
            distance_matrix, trips_df, config, device
        )
        
        metrics.update(additional_metrics)
        
        visualize_predictions(model, X_test, y_test, config['output_path'], device=device)
        
        print("Model training and evaluation completed")
    
    # Generate predictions if requested
    if mode == 'predict':
        print("Loading pre-trained model for prediction...")
        
        model_path = os.path.join(config['output_path'], 'best_model.pth')
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model file not found: {model_path}")
        
        od_matrices = create_od_matrices(trips_df, station_mapping, config['num_stations'])
        X, y = create_features_and_targets(od_matrices, station_data, distance_matrix, config)
        
        scaler_X = StandardScaler()
        X_scaled = scaler_X.fit_transform(X)
        
        _, X_test, _, y_test = train_test_split(
            X_scaled, y, test_size=0.2, random_state=42
        )
        
        checkpoint = torch.load(model_path, map_location=device)
        
        model = CitibikeODModel(
            checkpoint['config'], X_scaled.shape[1], adjacency_matrix
        )
        model.to(device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        
        print("Evaluating model on test data...")
        metrics = evaluate_model(model, X_test, y_test, device)
        
        print("Running extended model evaluation...")
        feature_names = create_feature_names()
        
        additional_metrics = extended_evaluate_model(
            model, X_test, y_test, station_data, station_mapping, 
            distance_matrix, trips_df, config, device
        )
        
        metrics.update(additional_metrics)
        
        visualize_predictions(model, X_test, y_test, config['output_path'], device=device)
    
    print("Analysis completed successfully")
    return True

def main():
    """Main entry point for Citibike OD Matrix Analysis"""
    # Configuration variables from RUNTIME_CONFIG
    DATA_FILES_PATTERN = RUNTIME_CONFIG['DATA_FILES_PATTERN']
    DATA_DIR = RUNTIME_CONFIG['DATA_DIR']
    OUTPUT_DIR = RUNTIME_CONFIG['OUTPUT_DIR']
    MODE = RUNTIME_CONFIG['MODE']
    
    # Update config
    config = CITIBIKE_CONFIG.copy()
    config['output_path'] = OUTPUT_DIR
    config['data_path'] = DATA_DIR
    
    # Construct full data file pattern
    full_pattern = os.path.join(DATA_DIR, DATA_FILES_PATTERN)
    
    print(f"Starting Citibike OD Analysis")
    print(f"Data files pattern: {full_pattern}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Mode: {MODE}")
    
    try:
        # Process data and run analysis
        process_citibike_data(full_pattern, config, MODE)
        return 0
    except FileNotFoundError as e:
        print(f"File not found: {e}")
        return 1
    except MemoryError as e:
        print(f"Memory error: {e}")
        print(f"Error: Out of memory. Try reducing the sample rate or number of stations.")
        return 1
    except Exception as e:
        print(f"Error during analysis: {e}")
        return 1

if __name__ == "__main__":
    exit(main())

Starting Citibike OD Analysis
Data files pattern: data/citibike\JC2024*citibiketripdata.csv
Output directory: results/citibike_od_annual
Mode: visualize
Using device: cuda
Loading Citibike data with 100.0% sampling rate
Found 12 files to process
Processing file: JC202401citibiketripdata.csv
Extracted month: 1
Loaded 50661 sampled trips from JC202401citibiketripdata.csv
Processing file: JC202402citibiketripdata.csv
Extracted month: 2
Loaded 55613 sampled trips from JC202402citibiketripdata.csv
Processing file: JC202403citibiketripdata.csv
Extracted month: 3
Loaded 65581 sampled trips from JC202403citibiketripdata.csv
Processing file: JC202404citibiketripdata.csv
Extracted month: 4
Loaded 79116 sampled trips from JC202404citibiketripdata.csv
Processing file: JC202405citibiketripdata.csv
Extracted month: 5
Loaded 97479 sampled trips from JC202405citibiketripdata.csv
Processing file: JC202406citibiketripdata.csv
Extracted month: 6
Loaded 111115 sampled trips from JC202406citibiketripdata.c